# Preprocessing: Regional aggregation

This step of the tutorial is only needed if you want to emulate regionally-aggregated indicators, and can be skipped if you want gridcell-based emulation instead. In this step, we use region masks to spatially aggregate our indicator. Those masks should have the same resolution as the indicator, and give a weight for each gridpoint. This weight can be based on area, population, or any other feature relevant to the indicator.

In this tutorial, we'll create continental aggregations of the heating-degree-days indicator from the previous tutorial (`001_preprocessing_data_setup.ipynb`), using the continental masks already prepared in `data/continental_masks`. We use latitude weighting here, but in principle any weighting can be applied (e.g. population weighting, GDP weighting, harvest area, or anything else available that makes sense for the indicator in question).

First we need to (re-)define our indicator and paths, as done in `001_preprocessing_data_setup.ipynb`. Usually you'd set these directly in `config.toml`; we set them here in code instead, purely so the tutorial is self-contained and doesn't require editing the config file.

In [1]:
import xarray as xr
from rimeX.config import CONFIG 

CONFIG['isimip.download_folder'] = 'data/downloads'
CONFIG['indicators.folder'] = 'data/indicators'
CONFIG['isimip.climate_impact_explorer'] = 'data'
CONFIG['indicator.heating-degree-days'] = {
    'frequency': 'annual',
    'units': 'days*°C',
    'isimip_meta': {
        'db_file': 'data/heating-degree-days.json'
    }
}

Now we define the configs specific to this step: the mask folder, and which weighting(s) to use.

Each mask file follows the structure `{masks_folder}/{region}/masks/{region}_{resolution_information}_{weighting}.nc4`, and contains one or more data variables — one per subregion (e.g. `AFRICA` for the full continent and a country level split) — each a 2D `(lat, lon)` array on the same grid as the indicator it will be applied to (Note that the included masks have a 5 degree resolution for storage considerations and will thus only be applicable to the `heating-degree-days` indicator). The value at each gridpoint is the weight to give that point when aggregating: for `latWeight`, this is a fractional weight — the fraction of the gridcell covered by the region, scaled by the cosine of latitude (to account for gridcells shrinking in area toward the poles), with `0` outside the region. Other weighting schemes (population, GDP, harvest area, etc.) follow the same shape and convention, just with a different per-gridpoint weight in place of the area weighting.

`run_regional_averages` takes a **weighted average** (or weighted sum, depending on `aggregation_type`) of the indicator over each region's nonzero weights — so the mask isn't a simple 0/1 selector, it directly (and proportionally, for gridcells only partially inside the region) determines how much each gridpoint contributes to the region's aggregate value.

In [2]:
CONFIG['preprocessing.regional.weights'] = ["latWeight"]
CONFIG['preprocessing.regional.masks_folder'] = 'data/continental_masks'

## Running the regional aggregation

`run_regional_averages` computes, for every simulation of the given indicator(s), a weighted average of the indicator within each region:

```python
run_regional_averages(
    indicator,                    # list[str]: indicator name(s) to process
    aggregation_type="mean",      # "mean" or "sum" -- how gridpoints within a region are combined
    overwrite=False,              # recompute even if output files already exist
    region=None,                  # list of regions to process (default: all regions found under masks_folder)
    weights=None,                 # list of weighting schemes to use (default: preprocessing.regional.weights)
    cpus=None,                    # number of parallel worker processes (default: run sequentially)
    model=None,                   # filter by climate model (default: all)
    experiment=None,              # filter by climate scenario (default: all)
    impact_model=None,            # filter by impact model, for indicators with an ensemble dimension (default: all)
)
```

For each (indicator, simulation) pair (and possibly impact_model), it writes one CSV per region+weighting combination, plus one merged CSV per weighting that combines all regions into a single file with one column per region -- that's the "regional averages" file referenced in the log output below.

**From the command line:** the same computation is available as a CLI tool once your indicators are defined in `config.toml`, e.g.:

```bash
rime-regional-averages -i heating-degree-days -a mean --weights latWeight
```

This accepts the same options as above (`--overwrite`, `--region`, `--weights`, `--cpus`, plus the standard model/experiment filters), so you can run this step outside a notebook once you've moved your indicator definitions into the config file.

In [3]:
from rimeX.preproc.regional_average import run_regional_averages
run_regional_averages(["heating-degree-days"])

[14:03:29 | rimeX | INFO] Found 16 simulations for heating-degree-days
[14:03:29 | rimeX | INFO] After filtering, 16 simulations remain for heating-degree-days
[14:03:29 | rimeX | INFO] heating-degree-days, {'climate_scenario': 'historical', 'climate_forcing': 'gfdl-esm4'}:: process 8 region-mask
[14:03:29 | rimeX | INFO] heating-degree-days|{'climate_scenario': 'historical', 'climate_forcing': 'gfdl-esm4'}|latWeight : write merged regional averages to data/indicators/heating-degree-days/historical/gfdl-esm4/gfdl-esm4_historical_heating-degree-days_regional_latweight_annual_1850_2014.csv
[14:03:29 | rimeX | INFO] heating-degree-days, {'climate_scenario': 'historical', 'climate_forcing': 'mpi-esm1-2-hr'}:: process 8 region-mask
[14:03:29 | rimeX | INFO] heating-degree-days|{'climate_scenario': 'historical', 'climate_forcing': 'mpi-esm1-2-hr'}|latWeight : write merged regional averages to data/indicators/heating-degree-days/historical/mpi-esm1-2-hr/mpi-esm1-2-hr_historical_heating-degree

## Adding already aggregated indicators to RIME-X

For each (indicator, simulation, region, weighting) combination, the aggregation code writes one CSV following the structure:

```
{indicators.folder}/{indicator}/{scenario}/{model}/{weighting}/{region}/{model}_{scenario}_{indicator}_{region}_{weighting}_{frequency}_{start_year}_{end_year}.csv
```

e.g. `data/indicators/heating-degree-days/ssp126/mpi-esm1-2-hr/latWeight/AFR/mpi-esm1-2-hr_ssp126_heating-degree-days_afr_latweight_annual_2015_2100.csv`, and if an impact model is specified `{model}_{impact_model}_*.csv` with one column per subregion: 

```
time,AFR,AFRICA
2015-07-02,174.5806,174.5806
2016-07-02,178.67816,178.67816
...
```

**If you already have your own regionally-aggregated data** you can skip this whole aggregation step: just put the data in the same CSV format (a `time` column plus one column per region) and place your files in the same folder structure. Also write a config entry for the indicator, and build a database file for it exactly as we did for the gridded case in `001_preprocessing_data_setup.ipynb` -- except this time you can leave the `path` field in each db entry empty, since RIME-X derives the actual file path from the specifiers rather than from that field at this stage. You still need to fill in the rest of the specifiers (`climate_forcing`, `climate_scenario`, `simulation_round`, `time_step`, etc.) so RIME-X can identify each simulation correctly. This lets RIME-X pick up already-aggregated indicators at this step, without ever needing the underlying gridded data at all.

## Next steps

At this point, `heating-degree-days` has been regionally aggregated into per-region time series for each simulation, ready for the next stage of preprocessing -- computing quantile maps. See `003_preprocessing_quantilemaps.ipynb` in this tutorial series for that step.

Once the quantile maps are obtained, you can use SCM (Simple Climate Model) data for your scenarios of interest to run emulations -- see `004_emulations.ipynb`. If you still need to obtain SCM data for your scenarios of interest, see `005_run_FaIR.ipynb` first.

## Appendix: additional regional aggregation config options

The example above only sets `preprocessing.regional.weights` and `preprocessing.regional.masks_folder`, and calls `run_regional_averages` with just an indicator list. A few more options are available for more complex setups (see `[preprocessing.regional]` in `config.toml` and `rimeX/preproc/regional_average.py` for the underlying logic):

**Config keys**

| Key | Type | Description |
|---|---|---|
| `preprocessing.regional.masks_folder` | str | Root folder containing region masks, structured as `{masks_folder}/{region}/masks/{region}_{resolution_info}_{weighting}.nc4`. Which regions exist is auto-discovered from this folder's subdirectories (`get_all_regions()`), so adding a new region is just a matter of adding a new subfolder with mask files in it. |
| `preprocessing.regional.weights` | list[str] | The default set of weighting schemes applied when aggregating regions, e.g. `["latWeight", "gdp2020", "pop2020", ...]`. Any weighting used here needs a matching mask file (`..._{weighting}.nc4`) for every region you want to aggregate. |
| `spatial_aggregation` (per-indicator) | list[str] | Overrides `preprocessing.regional.weights` for a *specific* indicator, e.g. `river_discharge` and `runoff` in `config.toml` restrict themselves to `["latWeight", "pop2015"]` rather than the full global weights list -- useful when only some weightings make physical sense for a given indicator. |

**`run_regional_averages` / CLI options not used above**

| Option | Description |
|---|---|
| `aggregation_type` (`-a`) | `"mean"` (weighted average, the default) or `"sum"` (weighted sum) -- e.g. population-weighted *sum* rather than mean might be more appropriate for an indicator you want expressed as an absolute total rather than an intensity. |
| `overwrite` | Recompute and overwrite existing regional CSVs, rather than skipping simulations that already have output. |
| `region` | Restrict processing to a subset of regions instead of all regions found under `masks_folder`. Note: if you restrict to a subset, the merged all-regions CSV is skipped (since it wouldn't actually contain all regions). |
| `cpus` | Number of parallel worker processes; each (indicator, simulation) pair is processed as one job, so this parallelizes across simulations. |
| `model`, `experiment`, `impact_model` | Filter which simulations get processed, e.g. to only regionally-aggregate a single model/scenario rather than the full ensemble. |

For most use cases, setting `preprocessing.regional.weights` and `preprocessing.regional.masks_folder` (or their per-indicator `spatial_aggregation` override) is enough -- the options above mainly matter once you're running this at scale across many indicators, models, and weighting schemes.